Typed `@formatter` Decorator
==============================================================================

Note created when refactoring the text formatting stuff in `splatlog.lib` to
support proper type hints and suggestions, though focus ended up being on the 
subsequent work to improve the fallback/any formatting with `rich.pretty`.

In [1]:
import splatlog as slog
from splatlog.lib.text import fmt
from splatlog.reporting import Report

r = Report(console=slog.rich.to_console())

print(fmt(r))

Report(
    console=<console width=115 ColorSystem.TRUECOLOR>,
    include='all',
    show_placeholder_loggers=False,
    show_null_handlers=False
)


`fmt` type/value formatting used to be more-or-less like:

```md
TypeError: Expected `RelationshipProperty[Any].target` to be `sqlalchemy.sql.schema.Table`, given `{str: int}`: `{"x": 123, "y": 456, "z": 789}`
```

which would render with Markdown as:

> TypeError: Expected `RelationshipProperty[Any].target` to be `sqlalchemy.sql.schema.Table`, given `{str: int}`: `{"x": 123, "y": 456, "z": 789}`

Now, with `<>`-quoting types and type hints and using `Exception.add_note` it
will be 

```md
TypeError: expected `<RelationshipProperty[Any].target>` to be `<sqlalchemy.sql.schema.Table>`
given `<{str: int}>` `{"x": 123, "y": 456, "z": 789}`
```

Rendering as Markdown, I expect the notes would be included as a list like:

> **_TypeError:_** expected `<RelationshipProperty[Any].target>` to be `<sqlalchemy.sql.schema.Table>`
>
> -   given `<{str: int}>` `{"x": 123, "y": 456, "z": 789}`

In [2]:
reducer = slog.json.JSONReducer(
    name="match_fail",
    priority=1,
    is_match=lambda obj: obj.x == 1,
    reduce=lambda obj: obj.__dict__,
)

encoder = slog.json.JSONEncoder()
# encoder.add_reducers(reducer)
encoder.on_reducer_error = "continue"

encoder.encode(r)

'{"__class__": "splatlog.reporting.Report", "__repr__": "Report(console=<console width=115 ColorSystem.TRUECOLOR>, include=\'all\', show_placeholder_loggers=False, show_null_handlers=False)"}'

In [3]:
import sys


encoder.encode(sys.stderr)

'{"__class__": "ipykernel.iostream.OutStream", "__repr__": "<ipykernel.iostream.OutStream object at 0x110ef8670>"}'

In [4]:
slog.setup(
    level=slog.DEBUG,
    console=True,
    theme=slog.rich.THEME_ANSI_DARK,
)

log = slog.getLogger(__name__)


def f():
    err = AttributeError("hey")
    err.add_note("ho\nho\nho")
    err.add_note("let's go!")
    raise err


try:
    f()
except Exception as err:
    log.error("blah", exc_info=err)
    print(slog.lib.fmt(err))

ERROR       __main__                                                                                               
msg         blah                                                                                                   
err         ╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮   
            │ in <module>:18                                                                                   │   
            │                                                                                                  │   
            │   15                                                                                             │   
            │   16                                                                                             │   
            │   17 try:                                                                                        │   
            │ ❱ 18 │   f()                                                                                     │   
            │   19 except Exception as err:                                                                    │   
            │   20 │   log.error("blah", exc_info=err)                                                         │   
            │   21 │   print(slog.lib.fmt(err))                                                                │   
            │                                                                                                  │   
            │ in f:14                                                                                          │   
            │                                                                                                  │   
            │   11 │   err = AttributeError("hey")                                                             │   
            │   12 │   err.add_note("ho\nho\nho")                                                              │   
            │   13 │   err.add_note("let's go!")                                                               │   
            │ ❱ 14 │   raise err                                                                               │   
            │   15                                                                                             │   
            │   16                                                                                             │   
            │   17 try:                                                                                        │   
            ╰──────────────────────────────────────────────────────────────────────────────────────────────────╯   
            AttributeError: hey                                                                                    
            [NOTE] ho                                                                                              
            ho                                                                                                     
            ho                                                                                                     
            [NOTE] let's go!

Traceback (most recent call last):
  File "/private/tmp/ipykernel_98430/2031149008.py", line 18, in <module>
    f()
    ~^^
  File "/private/tmp/ipykernel_98430/2031149008.py", line 14, in f
    raise err
AttributeError: hey
ho
ho
ho
let's go!



```py
Traceback (most recent call last):
  File "/private/tmp/ipykernel_16101/2031149008.py", line 18, in <module>
    f()
    ~^^
  File "/private/tmp/ipykernel_16101/2031149008.py", line 14, in f
    raise err
AttributeError: hey
ho
ho
ho
let's go!
```

**Traceback (most recent call last):**

- **File:** `example.py`  
  **Line:** `10`  
  **Function:** `<module>`  
  **Code:** `main()`

- **File:** `example.py`  
  **Line:** `6`  
  **Function:** `main`  
  **Code:** `1 / 0`

**Exception:** `ZeroDivisionError`  
**Message:** division by zero

#### Traceback
##### most recent call last

- File `example.py`, line `10`, in `<module>`  
  
  ```py
  main()
  ```

- File `example.py:6` in `main`  
  
  ```py
  1 / 0
  ```

**Exception:** `ZeroDivisionError`  
**Message:** division by zero